# AGAR-RL V11: engine profile, GPU profile, training

This notebook benchmarks the active Colab runtime, initializes V11 from **policy weights only** in the latest valid V10 MaskablePPO checkpoint, and trains toward 20,000,000 V11 timesteps. V11 optimizer, timestep counter, VecNormalize statistics, and opponent league start fresh. If a V11 manifest already exists, the notebook resumes that V11 run instead.

The benchmark reports engine ticks/s, real PPO environment steps/s, rollout geometry, and GPU memory. A100/L4 performance is measured in the runtime where training runs; it is not inferred from a fixed profile. Checkpoints and concise metrics sync to a separate V11 Drive folder. Interrupting the training cell sends SIGINT to the trainer, which saves and syncs before exiting.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO = '/content/agario'
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', 'clone', 'https://github.com/Albin0903/agario.git', REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', 'checkout', '-B', 'main', 'origin/main'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Repository synced and dependencies installed.')


## Runtime check and engine profile
The engine profile is CPU-side and separate from neural-network training. It includes multiple cells/player and reports the timed phase coverage so raw engine throughput is not confused with SB3 FPS.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

# Env subprocesses inherit these caps so BLAS/OpenMP never spawn nested pools.
for var in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS','NUMBA_NUM_THREADS'):
    os.environ[var] = '1'
os.environ['AGARIO_DISABLE_TENSORBOARD'] = '1'
from src.training.hardware import effective_cpu_count
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab (L4 or A100) before running V11 training.')
torch.set_num_threads(1)
gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024**3)
cpu_count = effective_cpu_count()
print(f'GPU: {gpu} | VRAM: {vram_gib:.1f} GiB | usable CPU workers: {cpu_count} (affinity/cgroup limited)')
print('Placement: Numba physics and environment workers run on CPU; PPO updates are benchmarked on CPU and CUDA.')

def repeated_json_profile(module, args, rate_key, repeats=3):
    runs=[]
    for repeat in range(repeats):
        result=subprocess.run([sys.executable,'-m',module,*args],check=True,text=True,capture_output=True)
        row=json.loads(result.stdout.strip().splitlines()[-1])
        runs.append(row)
        print(f'{module} run {repeat+1}/{repeats}: {row[rate_key]:.1f}')
    representative=sorted(runs,key=lambda row:row[rate_key])[len(runs)//2]
    representative['repeat_rates']=[row[rate_key] for row in runs]
    return representative

ENGINE_PROFILE=repeated_json_profile('src.analysis.profile_engine',
    ['--steps','4000','--warmup-steps','250','--json'],
    'measured_engine_steps_per_second')
print(f"Engine median run: {ENGINE_PROFILE['measured_engine_steps_per_second']:.1f} ticks/s; "
      f"phase coverage {ENGINE_PROFILE['phase_coverage_percent']:.1f}%")
for phase,percent in ENGINE_PROFILE['phase_percent'].items():
    print(f'{phase:24s} {percent:6.2f}%')
ENV_PROFILE=repeated_json_profile('src.analysis.profile_env',
    ['--steps','2000','--warmup-steps','200','--json'],
    'environment_steps_per_second')
print(f"Full AgarEnv median run: {ENV_PROFILE['environment_steps_per_second']:.1f} decisions/s "
      f"({ENV_PROFILE['physics_ticks_per_second']:.1f} engine ticks/s) with bots + observation")


## V10 source or V11 resume
On the first V11 run, choose the valid V10 MaskablePPO archive with the highest **internal** timestep counter. This skips stale `ppo_final.zip` files whose filename or manifest claims a later step than the counter stored in the archive. For later notebook sessions, the V11 manifest controls resume geometry and the trainer restores the V11 optimizer and normalization state.

In [ ]:
import json, re, zipfile
from pathlib import Path

DRIVE_V10 = Path('/content/drive/MyDrive/agario_rl_backup_v10')
DRIVE_V11 = Path('/content/drive/MyDrive/agario_rl_backup_v11')
DRIVE_V11.mkdir(parents=True, exist_ok=True)
V11_MANIFEST = DRIVE_V11 / 'v11_manifest.json'

def stored_timesteps(path):
    try:
        with zipfile.ZipFile(path) as zf:
            data = zf.read('data').decode('utf-8', errors='replace')
        match = re.search(r'\"num_timesteps\"\s*:\s*(\d+)', data)
        return int(match.group(1)) if match else -1
    except (OSError, KeyError, zipfile.BadZipFile):
        return -1

V11_RESUME = V11_MANIFEST.is_file() and (DRIVE_V11 / 'ppo_latest.zip').is_file()
V10_CHECKPOINT = None
if V11_RESUME:
    manifest = json.loads(V11_MANIFEST.read_text(encoding='utf-8'))
    if manifest.get('version') != 'v11':
        raise RuntimeError('Refusing to resume a non-V11 checkpoint from the V11 folder.')
    PROFILE = {k: int(manifest[k]) for k in ('n_envs', 'n_steps', 'batch_size', 'n_epochs')}
    PROFILE['device'] = manifest.get('device', 'cuda')
    print(f"Resume V11 at {manifest['timesteps']:,} / 20,000,000 steps; saved profile: {PROFILE}")
else:
    if not DRIVE_V10.is_dir():
        raise FileNotFoundError(f'V10 Drive folder not found: {DRIVE_V10}')
    archives = [p for p in DRIVE_V10.glob('*.zip') if p.stat().st_size > 1024]
    candidates = [(stored_timesteps(p), p) for p in archives]
    candidates = [(step, p) for step, p in candidates if step >= 0]
    if not candidates:
        raise FileNotFoundError('No readable V10 checkpoint with a stored PPO timestep was found.')
    v10_step, V10_CHECKPOINT = max(candidates, key=lambda item: item[0])
    print(f'V11 weights source: {V10_CHECKPOINT} (checkpoint counter {v10_step:,})')
    if 'v10_manifest.json' in [p.name for p in DRIVE_V10.iterdir()]:
        v10_manifest = json.loads((DRIVE_V10 / 'v10_manifest.json').read_text(encoding='utf-8'))
        manifest_step = int(v10_manifest.get('timesteps', v10_step))
        if manifest_step != v10_step:
            print(f'V10 manifest counter ({manifest_step:,}) differs; using the archive counter ({v10_step:,}).')


## Choose a measured PPO profile
Fresh V11 runs benchmark short, independent PPO runs from the same V10 policy weights. The profile keeps the optimizer settings fixed and compares CPU worker count, batch size, real end-to-end SB3 FPS, and peak GPU memory. The candidate worker count is capped at the CPU logical-core count, following SB3's guidance for compute-bound environments.

In [ ]:
import json, re, statistics, subprocess, sys
from pathlib import Path

if not V11_RESUME:
    # Benchmark the practical 4–16 worker range observed for CPU physics.
    # Keep the worker count below the affinity/cgroup budget, and use a small
    # CPU-only baseline so it cannot dominate the GPU profile sweep.
    max_profile_workers = min(cpu_count, 16)
    gpu_workers = sorted({max(1, min(n, max_profile_workers)) for n in (4, 8, 12, 16)})
    cpu_workers = sorted({max(1, min(n, max_profile_workers)) for n in (4, 8)})
    gpu_batches = [1024, 2048] if vram_gib >= 16 else [256, 512]
    profile_root = Path('/content/v11_profiles')
    raw_runs = []
    for profile_device, worker_candidates, batches in (
        ('cuda', gpu_workers, gpu_batches),
        ('cpu', cpu_workers, [1024]),
    ):
        for n_envs in worker_candidates:
            for batch in batches:
                for repeat_id in range(2):
                    run_dir = profile_root / f'{profile_device}_envs_{n_envs}_batch_{batch}_repeat_{repeat_id}'
                    cmd = [sys.executable, '-m', 'src.training.train_v11',
                           '--v10-checkpoint', str(V10_CHECKPOINT), '--profile-run',
                           '--n-envs', str(n_envs), '--n-steps', '2048',
                           '--batch-size', str(batch), '--n-epochs', '8',
                           '--total-timesteps', '20000000', '--save-dir', str(run_dir / 'ppo'),
                           '--history-dir', str(run_dir / 'pool'), '--device', profile_device]
                    result = subprocess.run(cmd, text=True, capture_output=True)
                    print(f'Profile {repeat_id+1}/2: device={profile_device} envs={n_envs} batch={batch}')
                    print(result.stdout[-1800:])
                    if result.returncode:
                        print(result.stderr[-2500:])
                        continue
                    match = re.search(r'PROFILE_RESULT steps_per_second=([0-9.]+) max_vram_mb=([0-9.]+)', result.stdout)
                    if match:
                        raw_runs.append({'device': profile_device, 'n_envs': n_envs, 'n_steps': 2048,
                                         'batch_size': batch, 'n_epochs': 8,
                                         'steps_per_second': float(match.group(1)),
                                         'max_vram_mb': float(match.group(2))})
    profile_results = []
    for device, worker_candidates, batches in (
        ('cuda', gpu_workers, gpu_batches),
        ('cpu', cpu_workers, [1024]),
    ):
        for n_envs in worker_candidates:
            for batch in batches:
                runs = [r for r in raw_runs if r['device'] == device and r['n_envs'] == n_envs
                        and r['batch_size'] == batch]
                if len(runs) != 2:
                    print(f'Skipping incomplete profile pair: {device}, envs={n_envs}, batch={batch}')
                    continue
                rates = [r['steps_per_second'] for r in runs]
                profile_results.append({'device': device, 'n_envs': n_envs, 'n_steps': 2048,
                    'batch_size': batch, 'n_epochs': 8,
                    'steps_per_second': statistics.median(rates),
                    'steps_per_second_runs': rates,
                    'max_vram_mb': max(r['max_vram_mb'] for r in runs)})
    if not profile_results:
        raise RuntimeError('No complete repeated device/worker/batch profile pair succeeded.')
    PROFILE = max(profile_results, key=lambda row: row['steps_per_second'])
    (DRIVE_V11 / 'v11_profile_results.json').write_text(json.dumps({
        'gpu': gpu, 'vram_gib': vram_gib, 'cpu_budget': cpu_count,
        'gpu_workers': gpu_workers, 'cpu_workers': cpu_workers,
        'gpu_batches': gpu_batches, 'engine_profile': ENGINE_PROFILE,
        'environment_profile': ENV_PROFILE, 'raw_runs': raw_runs,
        'candidates_median_of_two': profile_results, 'selected': PROFILE},
        indent=2), encoding='utf-8')
    print('Median-of-two device/worker/batch sweep selected by actual SB3 steps/s:', PROFILE)
else:
    profile_results = []
    print('Profile retained from the V11 manifest; no new optimizer profile is needed for resume.')


## Training telemetry and scheduled scenario checks

The training log and `metrics.jsonl` record a window every 100k agent steps: mass/peak mass, kills and pellets per 100k, deaths, episode length/survival rate, longest life, reward components, split actions/cells, split decisions followed by a kill within 30 decisions, unresolved split decisions, ejections, rolling FPS, CPU load, and GPU utilization/memory. Split usefulness is diagnostic only; it does not change reward.

Every 1M steps the trainer saves a checkpoint and evaluates it on the same five held-out seeds in each of three controlled scenarios: standard settings, half the pellets, and 150% of the bot count. Per-episode rows go to `scenario_evaluations.jsonl` and sync to Drive. An interrupted scenario evaluation resumes missing seeds before PPO continues. These stress cases are simulation tests, not official Agar.io modes.

The target is a cumulative 20,000,000 V11 timesteps. A fresh run imports only V10 policy weights; a resumed V11 run restores its optimizer and actual timestep counter. The first-run profile compares CPU and CUDA updates over two runs per worker count, capped by the effective CPU affinity/cgroup budget. Stop this cell to forward SIGINT and save/sync.


In [ ]:
import signal, subprocess, sys
from pathlib import Path

common = [sys.executable, '-m', 'src.training.train_v11',
          '--n-envs', str(PROFILE['n_envs']), '--n-steps', str(PROFILE['n_steps']),
          '--batch-size', str(PROFILE['batch_size']), '--n-epochs', str(PROFILE['n_epochs']),
          '--total-timesteps', '20000000', '--save-dir', '/content/checkpoints/v11',
          '--history-dir', '/content/checkpoints/v11/self_play_pool',
          '--backup-dir', str(DRIVE_V11), '--device', str(PROFILE['device'])]
if V11_RESUME:
    cmd = common + ['--resume-v11', 'auto']
else:
    cmd = common + ['--v10-checkpoint', str(V10_CHECKPOINT)]
print(f"Starting V11 on {PROFILE['device']} with {PROFILE['n_envs']} env workers; 100k telemetry / 1M scenario evaluations.")
proc = subprocess.Popen(cmd)
try:
    exit_code = proc.wait()
except KeyboardInterrupt:
    print('Forwarding stop request to V11 trainer...')
    proc.send_signal(signal.SIGINT)
    exit_code = proc.wait()
if exit_code != 0:
    raise RuntimeError(f'V11 trainer exited with code {exit_code}.')
print('V11 training stopped cleanly or reached its target.')


## Saved artifacts

`MyDrive/agario_rl_backup_v11/metrics.jsonl` contains 100k-step behavior/throughput windows; `scenario_evaluations.jsonl` contains five held-out episodes in each of three scenarios at every 1M milestone; `v11_profile_results.json` records the repeated CPU/CUDA and worker-count sweep. Physics and environment workers run on CPU; PPO update device is selected empirically for this Colab runtime.
